In [1]:
!pip install torch torchvision timm matplotlib

In [2]:
import torch 
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image 
import timm

In [3]:
t1 = timm.create_model('vit_base_patch16_224', pretrained=True)
t2 = timm.create_model('resnet50' , pretrained=True)
t1.eval()
t2.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act2): ReLU(inplace=True)
      (aa): Identity()
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     

In [4]:
student = timm.create_model('vit_base_patch16_224', pretrained=False)
student.train()
print(type(student))

<class 'timm.models.vision_transformer.VisionTransformer'>


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [6]:
image = Image.open('/home/boss/Downloads/pixelcut-export (4).jpeg')
image = transform(image).unsqueeze(0)

In [7]:
with torch.no_grad():
    t1_output = t1(image)
    t2_output = t2(image)
student_output = student(image)

In [8]:
cos = nn.CosineSimilarity(dim=1)
l1 = nn.SmoothL1Loss()

def distillation_loss(student_out, teacher_out):
    cos_loss = 1 - cos(student_out, teacher_out).mean()
    l1_loss = l1(student_out, teacher_out)
    return cos_loss + l1_loss

In [9]:
loss1 = distillation_loss(student_output, t1_output)
loss2 = distillation_loss(student_output, t2_output)

total_loss = loss1 + loss2

In [10]:
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

optimizer.zero_grad()
total_loss.backward()
optimizer.step()

In [12]:
print(student_output.shape)
print(t1_output.shape)
print(t2_output.shape)

torch.Size([1, 1000])
torch.Size([1, 1000])
torch.Size([1, 1000])


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [14]:
student = student.to(device)
teacher1 = t1.to(device)
teacher2 = t2.to(device)

In [15]:
image2 = Image.open('/home/boss/Downloads/pixelcut-export (4).jpeg')

In [17]:
torch.save(student.state_dict(), "student_model_version1.pth")